# Ashvini: train every model on Google Colab

Before you start: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

Everything (downloaded data, model cards, the exported model) is saved in your Google Drive under `MyDrive/ashvini/`. If Colab disconnects, run all again: downloads are skipped when the file is already there, and finished modules are skipped when their model card exists.

What gets downloaded automatically:

| Module | Data | How |
|---|---|---|
| collision, rl | ESA Collision Avoidance Challenge | Zenodo record 4463683 |
| kp | NASA OMNI hourly | NASA SPDF |
| debris | CelesTrak debris clouds | CelesTrak GP as CSV |
| orbit_error, manoeuvre | Space-Track element history | Needs a free Space-Track account (step 5) |

Upload these yourself to the Drive folders named in step 6 if you want them: ESA-ADB (telemetry), SPEED+ (pose), CHAMP/GRACE densities (density), SDO images (flare). Modules without data are skipped.

## 1. GPU check

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU. CPU still works, only slower for pose and flare."
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
BASE = Path("/content/drive/MyDrive/ashvini")
DATA, MODELS = BASE / "data", BASE / "models"
for p in (DATA, MODELS, DATA / "telemetry", DATA / "speedplus", DATA / "density", DATA / "flare"):
    p.mkdir(parents=True, exist_ok=True)
print("Saving to", BASE)

## 3. Get the code

Clones your GitHub repository. If you have not pushed it yet, upload `ashvini.zip` to `MyDrive/ashvini/` and the cell unzips it instead.

In [ ]:
import os, subprocess
REPO = "https://github.com/Mahendr99ar/ashvini.git"
os.chdir("/content")
if not Path("/content/ashvini").exists():
    if subprocess.run(["git", "clone", "-q", REPO]).returncode != 0:
        assert (BASE / "ashvini.zip").exists(), "Push the repo to GitHub or put ashvini.zip in MyDrive/ashvini/"
        subprocess.run(["unzip", "-q", str(BASE / "ashvini.zip"), "-d", "/content"], check=True)
os.chdir("/content/ashvini/ml")
!pip -q install -e ".[rl]"
print("ready")

## 4. Smoke test (about a minute)

Proves the setup works on synthetic data. These numbers mean nothing; they are not saved to Drive.

In [ ]:
!python -m ashvini_ml.train_all --synthetic --quick --out /tmp/smoke

## 5. Download the public data

In [ ]:
import glob, json, shutil, urllib.request, zipfile

def fetch(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print("have", dest.name); return dest
    print("downloading", dest.name)
    req = urllib.request.Request(url, headers={"User-Agent": "Ashvini training (Colab)"})
    with urllib.request.urlopen(req, timeout=600) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)
    return dest

# ESA Collision Avoidance Challenge: list the record's files and fetch them all
esa = DATA / "esa_cdm"; esa.mkdir(exist_ok=True)
record = json.load(urllib.request.urlopen("https://zenodo.org/api/records/4463683", timeout=60))
for f in record["files"]:
    path = fetch(f["links"]["self"], esa / f["key"])
    if path.suffix == ".zip":
        with zipfile.ZipFile(path) as z: z.extractall(esa)
# the dataset zip holds train_data.zip inside it, so unpack nested zips too
for inner in esa.rglob("*.zip"):
    if inner.parent != esa and not inner.with_suffix(".csv").exists():
        with zipfile.ZipFile(inner) as z: z.extractall(inner.parent)
print("files in esa_cdm:", [str(p.relative_to(esa)) for p in esa.rglob("*") if p.is_file()][:20])
train_csv = sorted(glob.glob(str(esa / "**" / "train_data.csv"), recursive=True)) or sorted(glob.glob(str(esa / "**" / "*train*.csv"), recursive=True))
print("ESA training file:", train_csv[0] if train_csv else "NOT FOUND, check the folder")

# NASA OMNI hourly, all years in one file
omni = fetch("https://spdf.gsfc.nasa.gov/pub/data/omni/low_res_omni/omni2_all_years.dat", DATA / "omni2_all_years.dat")

# CelesTrak debris clouds, merged into one CSV
import pandas as pd
debris_csv = DATA / "celestrak_debris.csv"
if not debris_csv.exists():
    groups = ["fengyun-1c-debris", "cosmos-2251-debris", "iridium-33-debris", "cosmos-1408-debris"]
    frames = [pd.read_csv(f"https://celestrak.org/NORAD/elements/gp.php?GROUP={g}&FORMAT=csv") for g in groups]
    pd.concat(frames).to_csv(debris_csv, index=False)
print("debris rows:", len(pd.read_csv(debris_csv)))

### Space-Track history (optional)

Make a free account at space-track.org. In Colab, open the key icon (Secrets) on the left and add `SPACETRACK_USER` and `SPACETRACK_PASS`, with notebook access on. Space-Track's terms do not allow republishing this data; it stays in your Drive and only model results leave it.

In [ ]:
history = DATA / "spacetrack_history.tle"
try:
    from google.colab import userdata
    user, pw = userdata.get("SPACETRACK_USER"), userdata.get("SPACETRACK_PASS")
except Exception:
    user = pw = None

if history.exists():
    print("have", history.name)
elif user and pw:
    import requests
    # stations (which reboost), Hubble, and Earth-observation satellites; 60 days of element sets each
    ids = "25544,48274,20580,39084,49260,40697,42063,39634,41335,43013,37849,27424,25994"
    with requests.Session() as s:
        s.post("https://www.space-track.org/ajaxauth/login", data={"identity": user, "password": pw}, timeout=60).raise_for_status()
        q = f"https://www.space-track.org/basicspacedata/query/class/gp_history/NORAD_CAT_ID/{ids}/EPOCH/>now-60/orderby/NORAD_CAT_ID,EPOCH asc/format/tle"
        r = s.get(q, timeout=300); r.raise_for_status()
        history.write_text(r.text)
    print("element sets:", history.read_text().count("\n") // 2)
else:
    print("No Space-Track secrets set: orbit_error and manoeuvre will be skipped.")

## 6. Your own uploads (optional)

Put these in Drive, then rerun from here:

- `MyDrive/ashvini/data/telemetry/channels.parquet` and `labels.csv` (ESA-ADB, exported as described in `ashvini_ml/telemetry/train.py`)
- `MyDrive/ashvini/data/speedplus/` with SPEED+'s `synthetic/` and `lightbox/` folders
- `MyDrive/ashvini/data/density/densities.csv` with columns time, alt_km, lat, lst, density
- `MyDrive/ashvini/data/flare/labels.csv` (file, time, label) and the `.npy` images next to it

In [ ]:
config = {}
if train_csv:
    config["collision"] = {"data": train_csv[0]}
    config["rl"] = {"cdms": train_csv[0], "steps": 200000}
if omni.exists():
    config["kp"] = {"omni": [str(omni)]}
if debris_csv.exists():
    config["debris"] = {"catalogue": str(debris_csv)}
if history.exists():
    config["orbit_error"] = {"history": str(history)}
    config["manoeuvre"] = {"history": str(history)}
if (DATA / "telemetry" / "channels.parquet").exists():
    config["telemetry"] = {"channels": str(DATA / "telemetry" / "channels.parquet"), "labels": str(DATA / "telemetry" / "labels.csv")}
if (DATA / "speedplus" / "synthetic").exists():
    config["pose"] = {"root": str(DATA / "speedplus"), "epochs": 30}
if (DATA / "density" / "densities.csv").exists():
    config["density"] = {"densities": str(DATA / "density" / "densities.csv"), "omni": [str(omni)]}
if (DATA / "flare" / "labels.csv").exists():
    config["flare"] = {"images": str(DATA / "flare"), "labels": str(DATA / "flare" / "labels.csv")}

(BASE / "data.json").write_text(json.dumps(config, indent=2))
print("Will train:", ", ".join(config) or "nothing yet")

## 7. Train

Resumable: modules whose card is already in Drive are skipped. To retrain one, delete its `.json` card from `MyDrive/ashvini/models/`.

In [ ]:
!python -m ashvini_ml.train_all --config {BASE}/data.json --out {MODELS} --skip-done

## 8. Read the results

Every card says whether the model beat its baseline. Report the numbers exactly as they are, including the ones that lost.

In [ ]:
rows = []
for p in sorted(MODELS.glob("*.json")):
    if p.name.endswith(".serving.json") or p.name == "index.json":
        continue
    c = json.loads(p.read_text())
    rows.append({"model": c["name"], "beats baseline": c["beats_baseline"], "baseline": c["baseline"], "data": c["data"]})
display(pd.DataFrame(rows))
for p in sorted(MODELS.glob("*.md")):
    print(p.read_text(), "\n" + "-" * 70)

## 9. Take the model to the API

Downloads one zip with the served model and every card. Unzip it into `backend/models/` in the repository, then set `ASHVINI_RISK_MODEL_PATH=models/collision-risk.onnx` on Render and as a repository variable for the refresh workflow. Only do this if `collision-risk` beats its baseline.

In [ ]:
import shutil
from google.colab import files
bundle = shutil.make_archive("/content/ashvini-models", "zip", MODELS)
files.download(bundle)